In [1]:
%pip install topologicpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## Import the needed libraries

In [3]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\tuemi\miniconda3\envs\genai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Check TopologicalPy version

In [4]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.29) is EQUAL TO the latest version available on PyPI.


## Set my renderer

In [4]:
renderer = "vscode"

## Transfer dictionaries by key

In [5]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


## Import the gallery floor plan

In [6]:
gallery = Topology.ByOBJPath(r"C:\Users\tuemi\Downloads\A2_GraphML_Chloe.obj")

## Show the geometry

In [7]:
Topology.Show(gallery,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## Create grid overlay

In [12]:
gallery_topology = Cluster.ByTopologies(gallery) if isinstance(gallery, list) else gallery
b_r = Wire.BoundingRectangle(gallery_topology)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width") or (xmax - xmin)
length = Dictionary.ValueAtKey(d, "length") or (ymax - ymin)
uRange = list(range(0, int(width)+3, 3))
vRange = list(range(0, int(length)+3, 3))

b_r_face = Face.ByWire(b_r)
grid = Grid.EdgesByDistances(b_r_face, clip=True, uRange=uRange, vRange=vRange)

## Show the geometry and the grid

In [13]:
Topology.Show(gallery_topology, grid,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## Slice the floor plan with the grid to create topologic shell

In [16]:
shell = Topology.Slice(gallery_topology, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

## Show result

In [17]:
Topology.Show(shell,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## Navigation and analysis graphs from the shell

In [18]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)

## Derive and store the analysis graph vertices

In [19]:
g_verts = Graph.Vertices(analysis_graph)

## Show the analysis graph

In [20]:
Topology.Show(analysis_graph,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)
              

## Graph Analysis - Minimum Spanning Tree

In [21]:
cc1 = CellComplex.Prism()
cc2 = Topology.Translate(cc1, 1.1, 0, 0)
cc3 = Topology.Translate(cc2, 1.1, 0, 0)
g1 = Graph.ByTopology(cc1)
g2 = Graph.ByTopology(cc2)
g3 = Graph.ByTopology(cc3)
g2 = Graph.MinimumSpanningTree(g2)
g3 = Graph.Complete(g3)
Topology.Show(g1, g2, g3,
              vertexSize=12,
              vertexColor="red",
              edgeColor="lightgrey",
              edgeWidth=4,
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)


In [22]:
dn1 = Graph.Density(g1)
dn2 = Graph.Density(g2)
dn3 = Graph.Density(g3)
print("Density 1:", dn1)
print("Density 2:", dn2)
print("Density 3:", dn3)

Density 1: 0.42857142857142855
Density 2: 0.25
Density 3: 1.0


In [23]:
dr1 = Graph.Diameter(g1)
dr2 = Graph.Diameter(g2)
dr3 = Graph.Diameter(g3)
print("Diameter 1:", dr1)
print("Diameter 2:", dr2)
print("Diameter 3:", dr3)

Diameter 1: 3
Diameter 2: 5
Diameter 3: 1


## Shortest Path

In [ ]:
import time
from collections import deque

# --- Find all connected components via BFS ---
def bfs_component(graph, start_v, visited_keys):
    component = []
    queue = deque([start_v])
    while queue:
        v = queue.popleft()
        k = (round(Vertex.X(v), 3), round(Vertex.Y(v), 3))
        if k in visited_keys:
            continue
        visited_keys.add(k)
        component.append(v)
        for nv in (Graph.AdjacentVertices(graph, v) or []):
            nk = (round(Vertex.X(nv), 3), round(Vertex.Y(nv), 3))
            if nk not in visited_keys:
                queue.append(nv)
    return component

all_verts      = Graph.Vertices(analysis_graph)
visited_global = set()
components     = []
for v in all_verts:
    k = (round(Vertex.X(v), 3), round(Vertex.Y(v), 3))
    if k not in visited_global:
        comp = bfs_component(analysis_graph, v, visited_global)
        components.append(comp)

largest_comp = max(components, key=len)
print(f"Components: {len(components)}, largest: {len(largest_comp)} vertices")

# --- Pick start/end from within the largest component ---
def vdist2(v, x, y):
    return (Vertex.X(v) - x)**2 + (Vertex.Y(v) - y)**2

start_vertex = min(largest_comp, key=lambda v: vdist2(v, xmin+2, ymax-2))
end_vertex   = min(largest_comp, key=lambda v: vdist2(v, xmax-2, ymin+2))
print(f"Start: ({round(Vertex.X(start_vertex),2)}, {round(Vertex.Y(start_vertex),2)})")
print(f"End:   ({round(Vertex.X(end_vertex),2)}, {round(Vertex.Y(end_vertex),2)})")

# --- Compute shortest path ---
start = time.time()
path_result = Graph.ShortestPath(analysis_graph, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print(f"Shortest Path Duration: {round(end-start, 2)}s — type: {type(path_result).__name__}")

# ShortestPath returns a Wire directly in topologicpy 0.9.x
if path_result is not None and path_result.__class__.__name__ == 'Wire':
    shortest_path = path_result
elif path_result is not None:
    path_edges    = Graph.Edges(path_result) or Topology.Edges(path_result) or []
    shortest_path = Wire.ByEdges(path_edges) if path_edges else None
else:
    shortest_path = None

# Straighten
straight_path = Wire.Straighten(shortest_path, host=gallery_topology) if shortest_path else None

if shortest_path:
    print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
if straight_path:
    print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))

if shortest_path:
    for edge in (Topology.Edges(shortest_path) or []):
        edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(["width", "color"], [7, "red"]))
if straight_path:
    for edge in (Topology.Edges(straight_path) or []):
        edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(["width", "color"], [7, "blue"]))

Components: 97, largest: 565 vertices
Start: (73.5, 156.0)
End:   (191.5, 106.0)
Shortest Path Duration: 0.4 seconds
Graph.Edges - Error: The input 'graph' is not a valid Graph. Returning [].


In [36]:
Topology.Show(gallery_topology, shortest_path, straight_path,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColorKey="color",
              edgeWidthKey="width",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)